# BCSC–Transformer 二分类与捷径学习研究

本 Notebook 比较三种来源隔离的二分类方案：

1. 统一预处理波形直接输入 Transformer；
2. BCSC 直达径、多径及关系显式特征输入 Transformer；
3. 仅 BCSC 直达径显式特征输入相同的特征 Transformer。

研究问题是：波形模型是否利用背景或多径形成捷径，以及高置信度多径的显式表征能否辅助仅直达径识别。

> BCSC 的7/8复现率表示算法稳定性，不是路径物理真实性的后验概率。负样本中的结果仅称主/次到达候选。


## 1. 固定环境与配置

所有计算使用项目规定的 `D:\Python_env\toothwhale\python.exe`。随机种子、数据来源、BCSC约束、五折划分和模型训练预算均在流水线配置中固定。


In [1]:
from pathlib import Path
import json
import sys

MODULE_DIR = Path(r"D:\Project_Github\Indo-Pacific-humpback-dolphin9_20260813")
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from bcsc_transformer_pipeline import CFG, run_pipeline

print(f"Python: {sys.executable}")
print(f"输出目录: {CFG.output_dir}")
print(f"配置哈希: {CFG.config_hash}")
print(f"BCSC原子长度: {CFG.atom_length}点 ({CFG.atom_duration_us:.1f} μs)")


Python: D:\Python_env\toothwhale\python.exe
输出目录: D:\Project_Github\Indo-Pacific-humpback-dolphin\19_20260813\outputs
配置哈希: 33ce6ffa3ab921bc
BCSC原子长度: 49点 (85.0 μs)


D:\Python_env\toothwhale\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 运行完整流水线

该单元格按四个总体阶段显示进度和ETA：数据读取与统一预处理、7000条BCSC及稳定性复核、路径特征提取、三模型五折训练。缓存按配置哈希保存，意外中断后可继续。


In [2]:
results = run_pipeline(CFG, force=False, quick=False)
results


总体进度 2/4：BCSC与8次稳定性复核: 0条 [00:00, ?条/s]

总体进度 2/4：BCSC与8次稳定性复核: 0条 [00:00, ?条/s]

{
  "mode": "full",
  "config_hash": "33ce6ffa3ab921bc",
  "data_validation": {
    "positive_count": 3500,
    "negative_count": 3500,
    "unique_sample_ids": true,
    "all_waveforms_finite": true,
    "all_sample_rates_correct": true,
    "all_sample_counts_correct": true,
    "minimum_negative_gap_s": 0.010444444444431156,
    "source_group_leakage_count": 0,
    "passed": true
  },
  "synthetic_test": {
    "status": "ok",
    "true_delay_ms": 0.3506944444444444,
    "estimated_delay_ms": 0.3506944444444444,
    "absolute_error_ms": 0.0,
    "passed": true
  },
  "bcsc_completed": 7000,
  "elapsed_s": 194.840340499999,
  "bcsc_success_rate": 0.9992857142857143,
  "high_confidence_multipath_count": 5237,
  "feature_count": 75,
  "report_path": "D:\\Project_Github\\Indo-Pacific-humpback-dolphin\\19_20260813\\outputs\\research_report.md"
}


{'mode': 'full',
 'config_hash': '33ce6ffa3ab921bc',
 'data_validation': {'positive_count': 3500,
  'negative_count': 3500,
  'unique_sample_ids': True,
  'all_waveforms_finite': True,
  'all_sample_rates_correct': True,
  'all_sample_counts_correct': True,
  'minimum_negative_gap_s': 0.010444444444431156,
  'source_group_leakage_count': 0,
  'passed': True},
 'synthetic_test': {'status': 'ok',
  'true_delay_ms': 0.3506944444444444,
  'estimated_delay_ms': 0.3506944444444444,
  'absolute_error_ms': 0.0,
  'passed': True},
 'bcsc_completed': 7000,
 'elapsed_s': 194.840340499999,
 'bcsc_success_rate': 0.9992857142857143,
 'high_confidence_multipath_count': 5237,
 'feature_count': 75,
 'report_path': 'D:\\Project_Github\\Indo-Pacific-humpback-dolphin\\19_20260813\\outputs\\research_report.md'}

## 3. 主要结果表

下列单元格读取五折汇总，展示三组主要实验结果。阈值只由各折验证集确定，置信区间按原始录音组Bootstrap计算。


In [3]:
import pandas as pd
from IPython.display import Markdown, display

summary = pd.read_csv(CFG.output_dir / "model_summary.csv")
display(summary)
display(Markdown((CFG.output_dir / "research_report.md").read_text(encoding="utf-8")))


,model,auroc_fold_mean,auroc_ci_low,auroc_ci_high,auprc_fold_mean,auprc_ci_low,auprc_ci_high,balanced_accuracy_fold_mean,balanced_accuracy_ci_low,balanced_accuracy_ci_high,f1_fold_mean,f1_ci_low,f1_ci_high,sensitivity_fold_mean,sensitivity_ci_low,sensitivity_ci_high,specificity_fold_mean,specificity_ci_low,specificity_ci_high
0,direct_multipath,0.940331,0.911164,0.947426,0.914345,0.877714,0.922578,0.885379,0.862252,0.906831,0.891369,0.868836,0.911827,0.941235,0.907531,0.964841,0.829523,0.795369,0.865513
1,direct_only,0.930088,0.906213,0.942920,0.898984,0.865081,0.913411,0.871967,0.848336,0.896894,0.879306,0.856754,0.900896,0.934369,0.895935,0.960104,0.809566,0.774055,0.849212
2,waveform,0.980335,0.971287,0.986368,0.977922,0.967106,0.984970,0.935452,0.918062,0.951145,0.936019,0.919512,0.952122,0.942914,0.917303,0.962179,0.927989,0.902112,0.949603


# BCSC–Transformer 二分类与捷径学习研究报告

## 摘要

本实验使用3500条中华白海豚click正样本和3500条来源匹配的困难负样本。所有样本统一从原始波形获得5761点、576 kHz片段，并执行相同预处理。数据按原始录音编号严格隔离为5折，避免同一录音跨训练、验证和测试集合。

BCSC处理成功率为 **99.93%**；正样本高置信度多径检出率为 **57.17%**，负样本稳定次到达候选检出率为 **92.46%**。这里的7/8复现率表示算法稳定性，不是物理路径为真的后验概率。

## 三组主要结果

| 模型 | AUROC均值 [录音组Bootstrap 95% CI] | AUPRC均值 | 平衡准确率均值 | F1均值 |
|---|---:|---:|---:|---:|
| 直达径+多径显式特征 Transformer | 0.9403 [0.9112, 0.9474] | 0.9143 | 0.8854 | 0.8914 |
| 仅直达径显式特征 Transformer | 0.9301 [0.9062, 0.9429] | 0.8990 | 0.8720 | 0.8793 |
| 波形端到端 Transformer | 0.9803 [0.9713, 0.9864] | 0.9779 | 0.9355 | 0.9360 |

![三模型ROC和PR](figures/model_roc_pr.png)

## 问题一：波形模型是否存在背景或多径捷径

波形模型完整输入五折平均AUROC为 **0.9803**；仅背景残差AUROC为 **0.9265**；移除高置信度多径后的AUROC为 **0.9803**；跨类别背景交换造成 **18.19%** 的预测翻转，平均绝对概率变化为 **0.1785**。

梯度×输入归因中，直达径、多径和背景区域的平均贡献比例分别为 **47.49%**、**7.74%** 和 **44.77%**。该归因仅作为反事实检验的辅助证据。

基于预注册反事实判据，当前结果提供了背景/多径捷径证据。该判断依赖反事实扰动而不是仅依赖注意力或显著性图。

![反事实AUROC](figures/counterfactual_auroc.png)

## 问题二：显式多径是否辅助仅直达径识别

“直达径+多径”相对“仅直达径”的汇总AUROC差值为 **+0.0067**，录音组Bootstrap 95% CI为 **[-0.0039, +0.0164]**。显式多径表征的AUROC点估计更高，但95%置信区间包含0，因此尚无统计上明确的总体AUROC增益证据。

附加消融同时报告保留多径存在/置信度掩码但清零数值特征，以及只在两类均检出稳定次到达的子集上的结果，用于区分路径特性、检出存在性和选择效应。详见 `feature_ablation_metrics.csv`。

## BCSC与数据质量控制

- 合成直达+延迟副本测试：真实时延 0.3507 ms，估计时延 0.3507 ms，绝对误差 0.0000 ms，结果：通过。
- 数据量：正样本 3500，负样本 3500，总计 7000。
- BCSC未静默删除任何记录；失败或直达径不稳定的状态完整保存在 `bcsc_paths.csv`。
- 单个样本最多保存1条满足7/8复现和±0.03 ms延迟匹配的次到达候选。

![BCSC稳定度与时延](figures/bcsc_confidence_delay.png)

## 限制

单通道盲卷积分解存在尺度、符号、整体时移及源—信道分配不唯一性。负样本中的稳定次到达只称为算法候选。结果适用于当前35个原始录音来源，外推到其他水域、设备或季节前需要独立外部验证。

## 可复现信息

- 随机种子：20260813
- 配置哈希：`33ce6ffa3ab921bc`
- Python：`D:\Python_env\toothwhale\python.exe`
- GPU：NVIDIA GeForce RTX 4060


## 4. 输出索引

- `dataset_manifest.csv`：7000条样本及来源隔离折分；
- `bcsc_paths.csv`：直达径、最多1条高置信度多径和失败状态；
- `explicit_features.csv`：路径及关系特征；
- `fold_metrics.csv` / `test_predictions.csv`：三模型五折指标与样本级预测；
- `counterfactual_metrics.csv`：波形捷径反事实；
- `feature_ablation_metrics.csv`：多径数值特征和检出子集消融；
- `research_report.md`：独立中文研究报告。
